In [1]:
from datatools import get_price, get_cyq, get_limit_list, get_moneyflow, get_rzrq, get_toplist, get_report_roll
import pandas as pd
import numpy as np
from cal_cne6_factor import calc_cne6_factors

In [2]:
allstocks = pd.read_csv('data/allstock.csv')
allstocks = allstocks[(allstocks['list_date'] < 20260101) & ~allstocks['ts_code'].str.contains('BJ')].ts_code.tolist()
start_date='2021-01-01'
end_date='2026-05-28'

In [ ]:
cyq = get_cyq(allstocks, start_date, end_date)
cyq['winner_rate'] = cyq['winner_rate']/100
moneyflow = get_moneyflow(allstocks, start_date, end_date)
rzrq = get_rzrq(allstocks, start_date, end_date)
other_factors = pd.merge(cyq, moneyflow, on=['ts_code', 'trade_date'], how='outer').merge(rzrq, on=['ts_code', 'trade_date'], how='outer')
other_factors.to_parquet('data/factor/other_factors.parquet',index=False)

In [ ]:
data = get_price(codes=allstocks, start_date=start_date, end_date=end_date, fields=['open'])
data['return_flag'] = ((data.groupby('ts_code')['open'].shift(-21) - data['open']) / data['open'] >= 0.3).astype(int)
price = data[data['trade_date'].between(start_date, '2023-12-31')]
price['return_flag'].mean()